In [47]:
from transformers import DataCollatorWithPadding
import torch
import torch.nn as nn
from transformers import Trainer
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import torch
from datasets import load_dataset
import os
from transformers import AutoTokenizer, EarlyStoppingCallback
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
from transformers import AutoConfig
from evaluate import load
import matplotlib.pyplot as plt
from torch import nn
import pandas as pd
from datasets import DatasetDict, Dataset
import time 
from safetensors.torch import load_file  # comes with HF if safetensors installed
from transformers import BatchEncoding
from sklearn.metrics import f1_score, precision_score, recall_score
from scipy.special import softmax

In [48]:

tik = time.time()

#### Run Params

train_full_model = True
all_labels = False
model_name = "ProsusAI/finbert"
model_name = "bert-base-uncased"
num_layers = 2

#####

mypath = "/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/"

#data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data_right_classifications/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx_with_null/"
data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx_with_no_class/"
data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/paragraph_and_sentence_len_4_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx/"
data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/test_read_training_data/paragraph_and_sentence_len_5_min_chunk_len_100_cos_thresh_0.4_nace_level_1_stoxx_with_no_class__cos_sim_04/"
data_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__sample_ratio_1__filter_only_right_chunks_labeling"
data_path = "../../data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__sample_ratio_1__filter_only_right_chunks_labeling"
data_path = mypath + "data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__sample_ratio_1__filter_only_right_chunks_labeling"
data_path = mypath + "data/training_data/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__sample_ratio_1__filter_only_right_chunks__with_null_classifiers__nace_level_1"
data_path = mypath + "data/training_data/dataset_reports_subset_from_full_data_1__sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_1__2nd_approach__nace_level_1__cos_thres_0.35"
data_path = mypath + "data/training_data/company_description_2"
data_path = mypath + "data/training_data/company_description_1"

new_thresh = 0.35

results_path = mypath + f"results/BERT_models/Relevancy_Classifier/results__new_approach_data__num_layers_{num_layers}__cos_thres_{new_thresh}" + os.path.basename(model_name)
#results_path = mypath + f"results/BERT_models/___results_null_classifiers__cos_thres_{new_thresh}__num_layers_{num_layers}" + os.path.basename(model_name)
if train_full_model: 
    results_path += "__train_full_model" 
else: 
    results_path += "__train_classifier_only" 

if all_labels: 
    results_path += "__all_labels" 
else: 
    results_path += "__some_labels" 
    
os.makedirs(results_path, exist_ok = True)

print(results_path)

/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/results/BERT_models/Relevancy_Classifier/results__new_approach_data__num_layers_2__cos_thres_0.35bert-base-uncased__train_full_model__some_labels


In [49]:
# Load a custom CSV file
data_files = {"train": os.path.join(data_path, "train_data.csv"), "test": os.path.join(data_path, "test_data.csv"), "validation": os.path.join(data_path, "val_data.csv")}
# Load the CSV files using pandas
train_df = pd.read_csv(data_files["train"])
test_df = pd.read_csv(data_files["test"])
validation_df = pd.read_csv(data_files["validation"])
validation_df.dtypes

text     object
label      bool
dtype: object

In [50]:
# Load a custom CSV file
data_files = {"train": os.path.join(data_path, "train_data.csv"), "test": os.path.join(data_path, "test_data.csv"), "validation": os.path.join(data_path, "val_data.csv")}
# Load the CSV files using pandas
train_df = pd.read_csv(data_files["train"])
test_df = pd.read_csv(data_files["test"])
validation_df = pd.read_csv(data_files["validation"])

if "report" in test_df.columns:
    train_df = train_df.drop(columns=["report", "Unnamed: 0"])
    test_df = test_df.drop(columns=["report", "Unnamed: 0"])
    validation_df = validation_df.drop(columns=["report", "Unnamed: 0"])

train_df = train_df.dropna()
validation_df = validation_df.dropna()
test_df = test_df.dropna()

In [51]:
# Load a custom CSV file
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df),
    "validation": Dataset.from_pandas(validation_df)
})

In [52]:
label_mapping = {char: val for val, char in enumerate(set(train_df["label"]))}

In [53]:
# Initialize the BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize a sample text
sample_text = dataset["train"][0]["text"]
tokens = tokenizer(sample_text, padding="max_length", truncation=True, max_length=512)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Apply the tokenizer to the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=False)

# Inspect tokenized samples
print(tokenized_datasets["train"][0])

Map:   0%|          | 0/694 [00:00<?, ? examples/s]

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

{'text': 'accellerons technology gives engines an extra boost in performance with the main purpose of improving their fuel efficiency and thus reducing their environmental impact by generating fewer emissions.', 'label': True, 'input_ids': [101, 16222, 24038, 5644, 2974, 3957, 5209, 2019, 4469, 12992, 1999, 2836, 2007, 1996, 2364, 3800, 1997, 9229, 2037, 4762, 8122, 1998, 2947, 8161, 2037, 4483, 4254, 2011, 11717, 8491, 11768, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [54]:
labels = dataset["train"]["label"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)

label2id = {char: val for val, char in enumerate(set(dataset["test"]["label"]))}
id2label = {val: char for val, char in enumerate(set(dataset["test"]["label"]))}

if model_name == "ProsusAI/finbert": 
    config = AutoConfig.from_pretrained(
        model_name, 
        num_labels=len(set(labels)), 
        problem_type="single_label_classification", 
        id2label=id2label,
        label2id=label2id
        )
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config, ignore_mismatched_sizes=True)
elif model_name == "bert-base-uncased": 
    config = AutoConfig.from_pretrained(
        model_name, 
        num_labels=len(set(labels)), 
        id2label=id2label,
        label2id=label2id
        )
    model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

hidden = 512

config.custom_hidden = hidden
config.custom_num_layers = num_layers

layers = []
for i in range(num_layers):
    in_dim = config.hidden_size if i == 0 else hidden
    layers.append(nn.Linear(in_dim, hidden))
    layers.append(nn.GELU())
    layers.append(nn.Dropout(0.2))

layers.append(nn.Linear(hidden, 1))

model.classifier = nn.Sequential(*layers)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [55]:
for param in model.bert.parameters():
    param.requires_grad = train_full_model
    #param.requires_grad = True # Train the wmbeddings as well!

# Keep only the classification head trainable
for param in model.classifier.parameters():
    param.requires_grad = True

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

# Define training arguments
training_args = TrainingArguments(
    output_dir=results_path,          # Directory for saving model checkpoints
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    learning_rate=5e-5,              # Start with a small learning rate
    per_device_train_batch_size=16,  # Batch size per GPU
    per_device_eval_batch_size=16,
    num_train_epochs=40,              # Number of epochs
    weight_decay=0.01,               # Regularization
    save_total_limit=1,              # Limit checkpoints to save space
    load_best_model_at_end=True,     # Automatically load the best checkpoint
    logging_dir="./logs",            # Directory for logs
    logging_strategy="epoch",            # Directory for logs
    logging_steps=100,               # Log every 100 steps
    #fp16=True,                      # Enable mixed precision for faster training
    save_strategy="epoch"
)

print(training_args)

Trainable parameters: 110139137
TrainingArguments(
_n_gpu=0,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strate

/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/nace_git/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [56]:
# Load a metric (F1-score in this case)
metric = load("f1")

# Define a custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="weighted")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator

DataCollatorWithPadding(tokenizer=BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')

In [57]:
#device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
device = torch.device("cuda") if torch.backends.mps.is_available() else torch.device("cpu")

In [58]:
import torch
import torch.nn as nn
from transformers import Trainer

class WeightedBCELossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)

        # Expecting something like [w_neg, w_pos] or a scalar for pos_weight
        if class_weights is not None:
            cw = torch.as_tensor(class_weights, dtype=torch.float32)

            if cw.numel() == 2:
                # e.g. class_weights = [w_neg, w_pos]
                # BCEWithLogitsLoss uses pos_weight (relative to negatives)
                pos_weight = cw[1] / cw[0]
            else:
                # If user already passed a single number, treat it as pos_weight directly
                pos_weight = cw.squeeze()

            self.pos_weight = pos_weight
        else:
            self.pos_weight = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # labels: [B], values {0, 1}
        labels = inputs["labels"].float()   # BCE needs float, not long

        # standard forward pass (without labels)
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits             # [B, 1] or [B]

        # Make sure shapes match: [B]
        logits = logits.view(-1)
        labels = labels.view(-1)

        # print("Logits: ", logits)
        # print("Labels: ", labels)
        
        # loss function for one logit with sigmoid
        loss_fn = nn.BCEWithLogitsLoss(
            pos_weight=self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        )
        loss = loss_fn(logits, labels)

        # print("Loss: ", loss)
        
        return (loss, outputs) if return_outputs else loss

In [95]:
# or load model 
ckpt =  mypath + "results/BERT_models/Relevancy_Classifier/relevancy_judge__2__num_layers_1__cos_thres_0.35__train_full_model_Truebert-base-uncased__train_full_model__all_labels_1/checkpoint-88"
ckpt =  mypath + "results/BERT_models/Relevancy_Classifier/relevancy_judge__2__num_layers_2__train_full_model_Truebert-base-uncased/checkpoint-88"
ckpt =  mypath + "results/BERT_models/Relevancy_Classifier/relevancy_judge__2__num_layers_1__train_full_model_Truebert-base-uncased/checkpoint-88"

def get_relevancy_model(ckpt_path: str): 
    # load checkpoint
    config = AutoConfig.from_pretrained(ckpt_path)
    
    # 2. Build a model from config (bare BertForSequenceClassification)
    model_binary = AutoModelForSequenceClassification.from_config(config)
    
    # 3. Rebuild the SAME classifier architecture as in training
    hidden = getattr(config, "custom_hidden", 512)  # fallback if not in config
    num_layers = getattr(config, "custom_num_layers", 1)
    
    layers = []
    for i in range(num_layers):
        in_dim = config.hidden_size if i == 0 else hidden
        layers.append(nn.Linear(in_dim, hidden))
        layers.append(nn.GELU())
        layers.append(nn.Dropout(0.2))
    layers.append(nn.Linear(hidden, 1))
    model_binary.classifier = nn.Sequential(*layers)
    
    # 4. Load weights from model.safetensors
    state_dict = load_file(os.path.join(ckpt_path, "model.safetensors"))
    model_binary.load_state_dict(state_dict, strict=True)  # will fail loudly if mismatch
    
    # 5. Inference mode
    model_binary.eval()
    
    return model_binary

model = get_relevancy_model(ckpt)

In [96]:
trainer = WeightedBCELossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    class_weights=class_weights,  # e.g. tensor([w_neg, w_pos])
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=2,
        early_stopping_threshold=0.0
    )]
)

/tmp/ipykernel_213465/548807802.py:7: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `WeightedBCELossTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [97]:
# batch = tokenized_datasets["train"][0:1]
# enc = BatchEncoding(batch)
# enc["labels"] = torch.Tensor(enc["label"])
# enc

In [98]:
#trainer.compute_loss(model, enc)

In [99]:
# Start trainingO
#trainer.train()

## Find threshold for classification

In [100]:
# Generate predictions
val_predictions = trainer.predict(tokenized_datasets["validation"])
# val_predictions = trainer.predict(tokenized_datasets["test"])

/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/nace_git/lib/python3.10/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [101]:
logits = val_predictions.predictions
probs = torch.sigmoid(torch.tensor(logits)).numpy()
true_labels = np.array(tokenized_datasets["validation"]["label"])

In [102]:
thresholds = np.linspace(0.01, 0.99, 99)
best_f1 = -1
best_t = None

for t in thresholds:
    preds = probs > t
    preds = preds.flatten()
    
    #precision = recall_score(true_labels, preds, average="macro")
    f1 = f1_score(true_labels, preds, average="macro")
    #precision = precision_score(true_labels, preds, average="macro")
    print(f1)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best macro precision:", best_f1)

model.config.threshold = best_t
#model.config.save_pretrained(trainer.args.output_dir)

0.2990936555891239
0.2990936555891239
0.2990936555891239
0.6613138686131387
0.716536025113254
0.7307577118897873
0.7352935675139816
0.748805256869773
0.7577384948161408
0.7886006508600651
0.7973799126637555
0.8060308789922523
0.8060020440397658
0.8060020440397658
0.8102884329467737
0.8102884329467737
0.8102884329467737
0.8145690440342757
0.8188443519000521
0.8188443519000521
0.823114830311483
0.81875
0.81875
0.8186285459012732
0.8228776790868294
0.827123695976155
0.8227192246761719
0.8227192246761719
0.8227192246761719
0.8227192246761719
0.8354241338112306
0.8354241338112306
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.835214953271028
0.8349805330937407
0.8349805330937407
0.8349805330937407
0.8349805330937407
0.8305078395743963
0.8260217472815898
0.8260217472815898
0.8260217472815898
0.830227976358

In [103]:
# # Evaluate the model
# results = trainer.evaluate()

# results_txt = str(results)

# print(results)

In [104]:
# Generate predictions
predictions = trainer.predict(tokenized_datasets["test"])
logits = predictions.predictions
probs = torch.sigmoid(torch.tensor(logits)).numpy()

/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis/nace_git/lib/python3.10/site-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [116]:
predicted_labels = probs > 0.5
predicted_labels = probs > 0.75
predicted_labels = probs > best_t

# Classification report
print(classification_report(tokenized_datasets["test"]["label"], predicted_labels))

              precision    recall  f1-score   support

       False       0.92      0.85      0.88       142
        True       0.79      0.88      0.83        90

    accuracy                           0.86       232
   macro avg       0.85      0.86      0.86       232
weighted avg       0.87      0.86      0.86       232



In [106]:
false_classified = ~(np.array(tokenized_datasets["test"]["label"]) == predicted_labels.flatten())

false_classified_df = pd.DataFrame(
    np.array([
        np.array(tokenized_datasets["test"]["label"])[false_classified], 
        predicted_labels[false_classified].flatten(), 
        np.array(tokenized_datasets["test"]["text"])[false_classified],        
        probs[false_classified].flatten(), 
    ]).T,
    columns=["label", "prediction", "text", "probabilities"])
        
false_classified_df["notes"] = None

In [107]:
false_classified_df

,label,prediction,text,probabilities,notes
0,False,True,entities within this area are primarily locate...,0.93229294,None
1,True,False,order backlog as of end of period meur,0.055155374,None
2,False,True,gt saphnelo approved in the eu as an addon the...,0.8253796,None
3,False,True,in fort worth texas andritz technology is effi...,0.81838495,None
4,True,False,employees as of end of period without apprentices,0.10195942,None
5,False,True,the proposed merger with aegon nederland is an...,0.8879403,None
6,False,True,we conduct annual internal audits throughout t...,0.94465506,None
7,True,False,bcv banque cantonale vaudoise was founded on d...,0.077267595,None
8,True,False,mortgage lead engine mle produkte zur baufinan...,0.045432907,None
9,False,True,neutral solutions those that meet regulatory s...,0.7972059,None


In [108]:
false_classified.shape

(232,)

In [109]:
np.array(tokenized_datasets["test"]["label"])[false_classified]

array([False,  True, False, False,  True, False, False,  True,  True,
       False, False, False, False, False,  True,  True, False, False,
       False,  True, False, False, False, False, False, False,  True,
        True,  True,  True, False, False])

In [110]:
results_txt += str(classification_report(tokenized_datasets["test"]["label"], predicted_labels))
results_txt += "\n\n\nTime: " + str(time.time() - tik)

# Confusion matrix
cm = confusion_matrix(tokenized_datasets["test"]["label"], predicted_labels, normalize="true")
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(label_mapping.keys()))
fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, xticks_rotation="vertical")
plt.show()
plt.savefig(results_path + "/conf_matrix.png")

with open(results_path + "/results_txt.txt", "w") as f:
    f.write(results_txt)

logs = pd.DataFrame(trainer.state.log_history)
logs.head()

plt.figure(figsize=(8,5))

plt.figure(figsize=(8,5))
logs_epoch = logs.dropna(subset="loss")
eval_loss_epoch = logs.dropna(subset="eval_loss")
plt.plot(logs_epoch["epoch"], logs_epoch["loss"], label="Training Loss")
if "eval_loss" in logs:
    plt.plot(eval_loss_epoch["epoch"], eval_loss_epoch["eval_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss per Epoch")
plt.legend()
plt.savefig(results_path + "/losses.png")

NameError: name 'results_txt' is not defined

In [ ]:
config = AutoConfig.from_pretrained("projects/nace_classification/nace_report_topic_analysis/results/BERT_models/Relevancy_Classifier/relevancy_judge__2__num_layers_2__train_full_model_Falsebert-base-uncased/checkpoint-880")